# Prophet Forecasting Playground

This notebook provides an interactive environment to test the cluster-based Prophet model in two different operational modes:
1. **Long-Term Forecast**: Relies solely on seasonal trends, holiday effects, and calendar features. Used for capacity planning.
2. **Day-Ahead Forecast**: Enriched with historical consumption lags (e.g., 24h and 1-week back). Used for short-term operational dispatching.

### Key Features
- **Dual-Mode Pipeline**: Test how adding consumption lags dramatically improves short-term accuracy.
- **Modular Architecture**: Uses decoupled functions from `src/models/prophet_model.py` for preprocessing, training, and inference.
- **Zero Data Leakage**: Scaling is strictly fitted on training data (< 2014).

## 0. Environment Setup

Resolve project-level modules from the `src` directory and load the necessary forecasting functions.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Ensure project root is in path
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Import specialized Prophet functions from correctly named module
from src.models.prophet_model import (
    load_processed_data, 
    preprocess_and_split, 
    train_models, 
    predict_models, 
    evaluate_models,
    save_prophet_artifacts
)
from src.tools.visualization import plot_cluster_portfolio, analyze_time_periods

print("✅ Setup complete. Local prophet modules loaded from src/.")

## 1. Global Data Loading

We load the global processed dataset once. It contains all features, including the raw consumption and weather signals that will be scaled differently depending on the forecast mode.

In [ ]:
dataset_path = os.path.join(PROJECT_ROOT, 'Datasets', 'processed_electricity_data.parquet')
df_long = load_processed_data(dataset_path)

print(f"\n Total observations loaded: {len(df_long):,}")

---

## 2. EXPERIMENT A: Long-Term Horizon (Weather Only)

In this mode, the model does NOT see the recent consumption lags. It learns to predict the load based strictly on the time of day, day of week, holidays, and external weather regressors.

In [13]:
# A.1 Prepare Data (Long Term)
train_agg_lt, test_agg_lt, test_raw_lt, scalers_lt, sw_lt, regs_lt = preprocess_and_split(df_long, mode='long_term')

# A.2 Train Cluster Models
models_lt = train_models(train_agg_lt, regs_lt)

# A.3 Predict & Un-scale
results_lt = predict_models(models_lt, test_agg_lt, test_raw_lt, scalers_lt, regs_lt)

# A.4 Evaluate
eval_lt, summary_lt = evaluate_models(results_lt)
display(summary_lt)

Training:  60%|██████    | 3/5 [02:33<01:42, 51.12s/it]


KeyboardInterrupt: 

In [ ]:
# Visualization for Long-Term results
plot_cluster_portfolio(eval_lt, summary_lt, model_label="Prophet (Long-Term)")

---

## 3. EXPERIMENT B: Day-Ahead Horizon (Weather + Consumption Lags)

In this mode, we provide Prophet with the consumption from 24h ago and 1 week ago. This allows the model to capture the "inertia" of the load, which is critical for precise day-ahead forecasts.

In [ ]:
# B.1 Prepare Data (Day Ahead)
train_agg_da, test_agg_da, test_raw_da, scalers_da, sw_da, regs_da = preprocess_and_split(df_long, mode='day_ahead')

# B.2 Train Cluster Models
models_da = train_models(train_agg_da, regs_da)

# B.3 Predict & Un-scale
results_da = predict_models(models_da, test_agg_da, test_raw_da, scalers_da, regs_da)

# B.4 Evaluate
eval_da, summary_da = evaluate_models(results_da)
display(summary_da)

In [ ]:
# Visualization for Day-Ahead results
plot_cluster_portfolio(eval_da, summary_da, model_label="Prophet (Day-Ahead)")

---

## 4. Comparison & Insights

Here we compare the performance indicators from both experiments. Typically, the inclusion of consumption lags in the **Day-Ahead** mode significantly reduces the Global WMAPE compared to the purely seasonal **Long-Term** model.

In [ ]:
comparison_df = pd.DataFrame({
    "Long-Term (Weather Only)": summary_lt["Portfolio_WMAPE"],
    "Day-Ahead (Weather + Lags)": summary_da["Portfolio_WMAPE"]
}).style.background_gradient(cmap='RdYlGn_r', axis=1)

print("Portfolio WMAPE Comparison by Cluster:")
display(comparison_df)

### Observations
- **Impact of Lags**: Compare how much the error decreases when adding `Lag_24h` and `Lag_1week` as regressors.
- **Stability**: Check which clusters show the greatest improvement (often industrial clusters or highly volatile profiles benefit more from lags).
- **Next Steps**: Now that the modular Prophet model is validated in both modes, we can save the artifacts for production inference.